In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

MCP_SERVER_URL="http://localhost:8000/sse"
OLLAMA_URL="http://localhost:11434"

In [2]:
client = MultiServerMCPClient(
    {
        "Demo": {
            "url": MCP_SERVER_URL,
            "transport": "sse",
        }
    }
)

In [3]:
###############################################
# Custom magic command to skip cells
# To use:
# %%skip

from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    """Skips the execution of the cell."""
    return

In [4]:
from langchain.globals import set_debug
set_debug(False)

In [5]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3:8b", base_url=OLLAMA_URL, reasoning=True, temperature=0)

In [6]:
tools = await client.get_tools()
print(tools)

prompt = """
You are a scheduling assistant. You must ONLY interact with the environment using the following tools:

- get_provider_names → returns a list of available providers
- get_provider_roles → returns a list of provider roles
- get_available_booking_slots_for_provider → returns list of 5 earliest available time slots for a provider. 
  response is JSON with a top-level "result" field.
  "result" is a list of dicts, each with:
    - slot_number (integer, sorted ascending)
    - provider_name (string)
    - time_slot (string in 'YYYY-MM-DD HH24:MI:SS' format).
- book_slot_for_provider → books a client appointment with a provider at a specified slot number.

STRICT RULES:
1. NEVER invent, summarize, analyze, or describe provider names, roles, or time slots. 
2. NEVER generate your own list. ONLY return exactly what the tool output provides inside the "result" field. 
3. If asked about availability, ALWAYS call get_available_booking_slots_for_provider once per request. 
4. When booking, ALWAYS use the exact slot_number and provider_name from the tool output.
5. You must not provide totals, summaries, statistics, distributions, or "observations" about slots. 
6. If asked for a specific slot (e.g. "what is the slot_number for 2024-08-20 10:00:00?"), 
   search the tool output JSON for that exact time_slot and return its slot_number exactly as-is.
"""

agent = create_react_agent(
    llm,
    tools, 
    prompt=prompt
)

[StructuredTool(name='get_provider_names', description='returns a list of available providers', args_schema={'properties': {}, 'title': 'get_provider_namesArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x114679800>), StructuredTool(name='get_provider_roles', description='returns a list of available provider roles', args_schema={'properties': {}, 'title': 'get_provider_rolesArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1146794e0>), StructuredTool(name='get_available_booking_slots_for_provider', description="\nReturns booking 5 earliest time slots for a provider.\n\nOutput is ALWAYS JSON with fields:\n  - slot_number (integer, sorted ascending)\n  - provider_name (string)\n  - time_slot (string, format 'YYYY-MM-DD HH24:MI:SS')\n", args_schema={'properties': {'provider_name': {'title': '

In [ ]:
#%%skip
#####################################################
# List of providers at clinic

resp = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What care providers are available in our clinic and what are their roles?"}]}
)
print(resp['messages'][-1].content)

In [ ]:
#%%skip
#####################################################
# List of open booking slots for provider at clinic
user_message = {
    "role": "user",
    "content": "I want to book an apointment with Susan Davis ? Find the 5 earliest times."
}

resp = await agent.ainvoke(
    {"messages": resp['messages'] + [user_message]}
)

# Print the agent's last message
print(resp['messages'][-1].content)

In [ ]:
#%%skip
#####################################################
# Book appointment

user_message = {
    "role": "user",
    "content": "I want to book an appointment for Susan Davis at slot_number slot 39 for client id 007"
}

resp = await agent.ainvoke(
    {"messages": resp['messages'] + [user_message]}
)

print(resp['messages'][-1].content)

In [ ]:
#%%skip
#####################################################
# List of open booking slots for provider at clinic - (The slot booked should no longer be in this list)

user_message = {
    "role": "user",
    "content": "I want to schedule an apointment with Susan Davis ? Find the 5 earliest times."
}

resp = await agent.ainvoke(
    {"messages": user_message}
)

# Print the agent's last message
print(resp['messages'][-1].content)